
# Multi-Band Optical Light Curve: FREDBlackbodyModel

This example demonstrates the :class:`~triceratops.models.generic.optical_photometry.FREDBlackbodyModel`
— the coupled model that pairs a Fast Rise, Exponential Decay (FRED) temporal
profile with a Planck blackbody SED, and convolves the result through a set of
photometric filters to produce broadband optical light curves.

The model evaluates

\begin{align}F_\mathrm{band}(t) =
    \phi_\mathrm{FRED}(t;\, t_0, \tau_r, \tau_d) \times
    \bigl[\mathbf{W}\, B_\nu(\boldsymbol{\nu};\, T_\mathrm{eff}, A)\bigr]_{\mathrm{band\_idx}}\end{align}

where:

- $\phi_\mathrm{FRED}$ is the dimensionless FRED temporal profile
  (peak value = 1, onset at $t_0$, rise timescale $\tau_r$,
  decay timescale $\tau_d$),
- $\mathbf{W}$ is the filter weight matrix from
  :class:`~triceratops.utils.phot_utils.FilterBundle` (shape
  ``(N_\mathrm{filters}, N_\mathrm{freq})``),
- $B_\nu$ is the Planck function, normalized so that the parameter
  $A$ equals $F_\nu$ at the V-band pivot frequency, and
- ``band_idx`` is an integer array that selects one filter per observation.

This architecture is described in detail in `optical_photometry_models`.

## See Also
- :class:`triceratops.models.generic.optical_photometry.FREDBlackbodyModel`
- :class:`triceratops.models.core.optical.CoupledOpticalModel`
- :class:`triceratops.utils.phot_utils.FilterBundle`
- `optical_light_curve` — data container for single-band optical time series
- Gallery: `sphx_glr_galleries_inference_plot_optical_lc_inference.py` — fitting this model to data


## Setup

We load the SDSS *ugriz* passbands from the ``speclite`` registry and assemble
them into a :class:`~triceratops.utils.phot_utils.FilterBundle`. The bundle
precomputes the weight matrix $\mathbf{W}$ once so that all subsequent
filter convolutions are a single matrix–vector multiply — the hot-loop pattern
used by the :class:`~triceratops.models.generic.optical_photometry.FREDBlackbodyModel`
internally.

See `sphx_glr_galleries_photometry_plot_filter_bundle.py` for a deeper
look at ``FilterBundle`` mechanics and a timing comparison.



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u

from triceratops.models.generic.optical_photometry import FREDBlackbodyModel
from triceratops.utils.phot_utils import FilterBundle, flux_to_ab_mag, load_filter_from_speclite
from triceratops.utils.plot_utils import set_plot_style

set_plot_style()

# Load SDSS ugriz passbands and give them short single-letter keys.
# The key names must match the ``band_name`` column used in the data container.
SDSS_BANDS = {
    "u": "sdss2010-u",
    "g": "sdss2010-g",
    "r": "sdss2010-r",
    "i": "sdss2010-i",
    "z": "sdss2010-z",
}

filters = {short: load_filter_from_speclite(speclite) for short, speclite in SDSS_BANDS.items()}
bundle = FilterBundle(filters)

print(bundle)
print(f"Filter names : {bundle.filter_names}")
print(f"Common grid  : {len(bundle.frequency_grid):,} frequency points")
print(f"Weight matrix: {bundle.weight_matrix.shape}")

## Build the Model and Set True Parameters

:class:`~triceratops.models.generic.optical_photometry.FREDBlackbodyModel` requires
a :class:`~triceratops.utils.phot_utils.FilterBundle` as its only constructor
argument.  The model's ``bundle`` attribute is also used later by
:meth:`~triceratops.data.optical_photometry.OpticalPhotometryContainer.to_inference_data`
to resolve ``band_name → band_idx`` automatically.

The five free parameters of this model are:

.. list-table::
   :header-rows: 1
   :widths: 20 15 65

   * - Parameter
     - Units
     - Meaning
   * - ``t_0``
     - s
     - Flare onset time
   * - ``tau_r``
     - s
     - Rise timescale
   * - ``tau_d``
     - s
     - Decay timescale
   * - ``T_eff``
     - K
     - Effective blackbody temperature (controls colour)
   * - ``amplitude``
     - erg/s/cm²/Hz
     - $F_\nu$ at V-band pivot at the temporal peak



In [ ]:
model = FREDBlackbodyModel(bundle)

# "True" parameters for the synthetic transient
TRUE_PARAMS = {
    "t_0": 1.0 * 86400,  # onset at 1 day [s]
    "tau_r": 2.0 * 86400,  # 2-day rise  [s]
    "tau_d": 10.0 * 86400,  # 10-day decay [s]
    "T_eff": 12_000.0,  # 12 000 K  [K]
    "amplitude": 5e-28,  # ~V-band peak flux [erg/s/cm²/Hz]
}

print(f"\nvariable_names : {model.variable_names}")
print(f"output_names   : {model.output_names}")

## Visualise the FRED Temporal Profile

Before looking at the multi-band result, it is instructive to inspect the
dimensionless FRED temporal normalization $\phi(t)$ that modulates the
SED amplitude:

\begin{align}\phi(t) =
    \begin{cases}
        0, & t < t_0 \\
        \left(1 - e^{-(t - t_0)/\tau_r}\right)
        e^{-(t - t_0)/\tau_d},
        & t \ge t_0
    \end{cases}\end{align}



In [ ]:
t_days = np.linspace(0, 50, 500)
t_s = t_days * 86400.0

t0_d = TRUE_PARAMS["t_0"] / 86400.0
tau_r_d = TRUE_PARAMS["tau_r"] / 86400.0
tau_d_d = TRUE_PARAMS["tau_d"] / 86400.0

dt = t_days - t0_d
phi = np.where(dt >= 0, (1 - np.exp(-dt / tau_r_d)) * np.exp(-dt / tau_d_d), 0.0)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(t_days, phi, color="black", lw=2)
ax.axvline(t0_d, color="steelblue", ls="--", label=rf"$t_0 = {t0_d:.0f}$ d")
ax.axvline(t0_d + tau_r_d, color="tomato", ls=":", label=rf"$t_0 + \tau_r = {t0_d + tau_r_d:.0f}$ d")
ax.axvline(t0_d + tau_d_d, color="seagreen", ls="-.", label=rf"$t_0 + \tau_d = {t0_d + tau_d_d:.0f}$ d")
ax.set_xlabel("Time [days]")
ax.set_ylabel(r"$\phi(t)$  [dimensionless]")
ax.set_title("FRED temporal normalization")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

The peak time of the FRED profile is determined implicitly by $\tau_r$
and $\tau_d$. A fast rise (small $\tau_r$) combined with a slow
decay (large $\tau_d$) gives an asymmetric profile characteristic of
optical transients such as novae and kilonovae.



## Multi-Band Light Curves in Flux

We evaluate the model at a dense time grid for each band by building
``(time, band_idx)`` variable arrays. The
:meth:`~triceratops.models.core.base.Model.forward_model` call performs:

1. Evaluate the Planck SED $B_\nu$ on ``bundle.frequency_grid``.
2. Convolve through the weight matrix → band-averaged fluxes (one per filter).
3. Multiply by the FRED normalization $\phi(t)$ per observation.



In [ ]:
BAND_COLORS = {"u": "#8b5cf6", "g": "#3b82f6", "r": "#22c55e", "i": "#f59e0b", "z": "#ef4444"}
band_list = bundle.filter_names  # ['u', 'g', 'r', 'i', 'z']

n_t = 300
t_fine = np.linspace(0.1, 50, n_t) * 86400.0  # seconds

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=False)

ax_flux, ax_mag = axes

for band_name in band_list:
    band_idx_val = bundle.filter_names.index(band_name)
    variables = {
        "time": t_fine,
        "band_idx": np.full(n_t, band_idx_val, dtype=int),
    }
    out = model.forward_model(variables, TRUE_PARAMS)
    flux = out.flux_density.value  # erg/s/cm²/Hz
    mag = flux_to_ab_mag(flux)

    t_plot = t_fine / 86400.0  # back to days for plotting
    color = BAND_COLORS[band_name]

    ax_flux.plot(t_plot, flux, color=color, lw=2, label=rf"${band_name}$")
    ax_mag.plot(t_plot, mag, color=color, lw=2, label=rf"${band_name}$")

ax_flux.set_xlabel("Time since reference [days]")
ax_flux.set_ylabel(r"$F_\nu$ [erg s$^{-1}$ cm$^{-2}$ Hz$^{-1}$]")
ax_flux.legend(ncol=2, fontsize=9)
ax_mag.set_xlabel("Time since reference [days]")
ax_mag.set_ylabel(r"AB magnitude")
ax_mag.set_title("AB magnitude")
ax_mag.invert_yaxis()
ax_mag.legend(ncol=2, fontsize=9)

plt.tight_layout()
plt.show()

Because the blackbody SED peaks near the ultraviolet at $T_\mathrm{eff}
= 12\,000$ K, the *u* band is brightest in absolute flux. In magnitude space
(inverted y-axis) the ordering reverses — fainter AB magnitudes correspond to
lower flux density in the redder bands.



## Colour Evolution

One of the strengths of the coupled model is that it naturally produces
**colour evolution**. Even though $\phi(t)$ is band-independent, the
SED shape is fixed (no spectral evolution in this model variant). The
instantaneous colour therefore depends only on the filter convolution of the
Planck function at the chosen temperature.

Here we compute the $g - r$ colour as a function of time and compare
it with the single-band peaks.



In [ ]:
band_g_idx = band_list.index("g")
band_r_idx = band_list.index("r")

flux_g = model.forward_model(
    {"time": t_fine, "band_idx": np.full(n_t, band_g_idx, dtype=int)}, TRUE_PARAMS
).flux_density.value

flux_r = model.forward_model(
    {"time": t_fine, "band_idx": np.full(n_t, band_r_idx, dtype=int)}, TRUE_PARAMS
).flux_density.value

# Only compute colour where flux is non-negligible (avoid log(0))
with_signal = flux_g > 0
mag_g = np.where(with_signal, flux_to_ab_mag(flux_g), np.nan)
mag_r = np.where(with_signal, flux_to_ab_mag(flux_r), np.nan)
g_minus_r = mag_g - mag_r

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

for band_name in ["u", "g", "r", "i", "z"]:
    idx = band_list.index(band_name)
    out = model.forward_model({"time": t_fine, "band_idx": np.full(n_t, idx, dtype=int)}, TRUE_PARAMS)
    ax1.plot(
        t_fine / 86400,
        flux_to_ab_mag(out.flux_density.value),
        color=BAND_COLORS[band_name],
        lw=2,
        label=rf"${band_name}$",
    )

ax2.plot(t_fine / 86400, g_minus_r, color="black", lw=2)
ax2.axhline(0, color="gray", ls="--", lw=0.8)

ax1.invert_yaxis()
ax1.set_ylabel("AB magnitude")
ax1.legend(ncol=5, fontsize=8)
ax1.set_title("Multi-band light curves")

ax2.set_xlabel("Time since reference [days]")
ax2.set_ylabel(r"$g - r$ [mag]")
ax2.set_title(r"Colour  ($g - r$)")

plt.tight_layout()
plt.show()

Because the temperature is fixed in this model variant, the $g - r$
colour is constant — it depends only on $T_\mathrm{eff}$ (and the filter
convolution), not on time. Models with spectral evolution (e.g. cooling or
shock reionization) would show colour evolution here. The
:class:`~triceratops.models.generic.optical_photometry.GeneralizedFREDBlackbodyModel`
follows the same pattern but with a more flexible temporal shape controlled by
additional shape exponents $\nu_r$ and $\nu_d$.



## Parameter Sensitivity

Below we sweep $T_\mathrm{eff}$ to show how the blackbody temperature
controls the $u - z$ colour span, illustrating the model's ability to
discriminate between hot (UV-bright) and cool (optically red) transients.



In [ ]:
t_s_single = np.full(1, 5.0 * 86400.0)  # t = 5 days, near peak

T_values = [6_000, 10_000, 20_000, 40_000]

fig, ax = plt.subplots(figsize=(8, 4))

for T_eff in T_values:
    params = {**TRUE_PARAMS, "T_eff": float(T_eff)}
    fluxes = []
    for band_name in band_list:
        idx = band_list.index(band_name)
        out = model.forward_model({"time": t_s_single, "band_idx": np.array([idx])}, params)
        fluxes.append(out.flux_density.value[0])

    fluxes = np.array(fluxes)
    mags = flux_to_ab_mag(fluxes)
    # Normalise to g-band for comparison
    mags_norm = mags - mags[band_list.index("g")]
    ax.plot(band_list, mags_norm, "o-", lw=2, label=rf"$T_\mathrm{{eff}} = {T_eff / 1000:.0f}$ kK")

ax.invert_yaxis()
ax.axhline(0, color="gray", ls="--", lw=0.8)
ax.set_xlabel("Band")
ax.set_ylabel(r"$m_\mathrm{band} - m_g$ [mag]")
ax.set_title(r"Colour relative to $g$-band as a function of $T_\mathrm{eff}$")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

A hotter blackbody (larger $T_\mathrm{eff}$) is bluer: its *u*-band
flux is enhanced relative to the red bands.  At $T_\mathrm{eff} \sim
6\,000$ K the SED peaks at red–optical wavelengths, making the transient
significantly redder than its hotter counterparts.  This colour sensitivity
allows Bayesian inference to constrain $T_\mathrm{eff}$ from
multi-band photometry even when individual bands are noisy.

To see how this model is fitted to data with MCMC, see
`sphx_glr_galleries_inference_plot_optical_lc_inference.py`.

